# Week 7 · 03 The variational autoencoder (VAE)

`02`'s autoencoder compresses well, but you can't use it to make new images: its codes sit in odd-shaped groups with gaps, and a random code usually decodes to a blob. The **variational autoencoder** (Kingma & Welling, 2014) changes two things: the encoder outputs a small cloud of codes instead of one point, and a second loss term pulls every cloud towards one standard normal distribution. Then every code near 0 decodes to something sensible, and new digits are one `torch.randn` away. This notebook has the most mathematics of the week: the reparameterization trick (§3), the KL term (§4, with its proof) and the ELBO (§5, with its proof).

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **generative model**: a model you can draw new examples from (new images that look like the training images), not just one that classifies or compresses.
- **prior $p(z)$**: the distribution codes should follow; for the VAE, the standard normal $\mathcal{N}(0, I)$: every coordinate centred at 0 with spread 1, independent.
- **$\mu$, $\log \sigma^2$**: what the VAE's encoder outputs for an image: the centre and the (log of the) spread of that image's cloud of codes, one pair per code dimension.
- **reparameterization trick**: drawing $z = \mu + \sigma \varepsilon$ with $\varepsilon \sim \mathcal{N}(0, I)$ instead of sampling $z$ directly, so that gradients can flow through $\mu$ and $\sigma$.
- **KL divergence**: a measure of how different two distributions are; 0 when they're equal (Week 2 · 02 §7).
- **ELBO**: evidence lower bound: a quantity that is always below the log-likelihood $\log p(x)$ and that the VAE maximizes instead of it.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from figures import *
import checks; checks.load(globals(), "03")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** The notebook in one picture.

In [ ]:
fig_vae_map()

| Section | What | Why it's here |
|---|---|---|
| 0 | recap, and booklet 9.4 (the EM M-step) from memory | EM is the other latent-variable method you know; §5 connects them |
| E | a trained VAE makes new digits | start from the end |
| 1 | the problem with `02`'s codes | why an autoencoder can't generate |
| 2 | the encoder outputs a distribution | the idea, the architecture, every number for one image |
| 3 | the reparameterization trick | how gradients get past the random step |
| 4 | the KL term | a proposition: its closed form, with the proof |
| 5 | the ELBO | why reconstruction + KL is the right loss, with the proof |
| 6 | training, and what the latent space looks like | the code that made `vae2.pt`; sampling; the manifold |
| 7 | exercises | the block's hand-written parts |

## 0. Recap first · 10 min, closed book

**Quick check:** KL divergence $D_{\text{KL}}(q \,\Vert\, p)$ for discrete distributions: the formula, and its two key properties?

<details><summary>Answer</summary>

$\sum_i q_i \log (q_i / p_i)$. It's $\ge 0$, and $= 0$ only when $q = p$. It isn't symmetric. [Week 2 · 02 §7 ↗](../week2/02_map_and_entropy_in_depth.ipynb)

</details>

**Quick check:** The density of $\mathcal{N}(\mu, \sigma^2)$, and its log?

<details><summary>Answer</summary>

$\frac{1}{\sqrt{2\pi\sigma^2}} e^{-(z - \mu)^2/(2\sigma^2)}$; log: $-\frac12 \log(2\pi\sigma^2) - \frac{(z-\mu)^2}{2\sigma^2}$. [Week 1 · 03 §3 ↗](../week1/03_math_essentials.ipynb)

</details>

**Quick check:** If $\varepsilon \sim \mathcal{N}(0, 1)$, what are the mean and variance of $\mu + \sigma\varepsilon$?

<details><summary>Answer</summary>

Mean $\mu$, variance $\sigma^2$: so $\mu + \sigma \varepsilon \sim \mathcal{N}(\mu, \sigma^2)$. §3 is built on this. [Week 1 · 03 §2 ↗](../week1/03_math_essentials.ipynb)

</details>

**Quick check:** In EM for a Gaussian mixture, what is the hidden (latent) variable, and what does the E-step compute?

<details><summary>Answer</summary>

Which component each point came from. The E-step computes each point's responsibilities $\gamma_{ik}$: the probability it came from component $k$, given the current parameters. [Week 4 · 04 §3 ↗](../week4/04_hierarchical_dbscan_gmm.ipynb)

</details>

**Quick check:** In `02` §4, what happened when a code from empty space was decoded, and why?

<details><summary>Answer</summary>

A blob: the AE was only trained on codes its training images produced, so nothing made other codes meaningful. [02 §4 ↗](02_autoencoders.ipynb)

</details>

**Exercise 1 · Booklet 9.4: the M-step.** Given `X` `(n, d)` and responsibilities `gamma` `(n, K)` (rows sum to 1), return `(pi, mu, Sigma)`: $N_k = \sum_i \gamma_{ik}$, $\pi_k = N_k/n$, $\mu_k = \sum_i \gamma_{ik} x_i / N_k$, $\Sigma_k = \sum_i \gamma_{ik}(x_i - \mu_k)(x_i - \mu_k)^\top/N_k + 10^{-6} I$. Shapes `(K,)`, `(K, d)`, `(K, d, d)`.

**Example**: what goes in, and what should come out:
```python
X = np.array([[0.0], [1.0], [9.0], [10.0]]); gamma = np.array([[1.0, 0], [1, 0], [0, 1], [0, 1]])
pi, mu, Sigma = gmm_m_step(X, gamma)
pi, mu.ravel(), Sigma.ravel().round(4)    # → (array([0.5, 0.5]), array([0.5, 9.5]), array([0.25, 0.25]))
```

In [ ]:
def gmm_m_step(X, gamma):
    pass  # your code

In [ ]:
check(1)

## E. Start from the end: a trained VAE makes new digits

`vae2.pt` is a VAE with a 2-number code, trained for ten epochs on MNIST by the code in §6. What it stores:

In [ ]:
X_tr, y_tr, X_te, y_te = mnist_tensors()        # (60000, 784) and (10000, 784), pixels in [0, 1]
vae = load_vae()
for name, t in vae.state_dict().items():
    print(name, tuple(t.shape))
# → encoder.0.weight (256, 784)
#   encoder.0.bias (256,)
#   to_mu.weight (2, 256)
#   to_mu.bias (2,)
#   to_logvar.weight (2, 256)
#   to_logvar.bias (2,)
#   decoder.0.weight (256, 2)
#   decoder.0.bias (256,)
#   decoder.2.weight (784, 256)
#   decoder.2.bias (784,)

Like `02`'s autoencoder, except that the encoder ends in **two** heads, `to_mu` and `to_logvar`. **Making new digits** needs only the decoder: draw codes from the standard normal and decode them.

In [ ]:
torch.manual_seed(0)
z = torch.randn(5, 2)                           # 5 codes, each 2 numbers from N(0, 1)
with torch.no_grad():
    imgs = vae.decoder(z)                       # (5, 784): five new images
print(z.numpy().round(2))
print(imgs.shape)
# → [[ 1.54 -0.29]
#     [-2.18  0.57]
#     [-1.08 -1.4 ]
#     [ 0.4   0.84]
#     [-0.72 -0.4 ]]
#   torch.Size([5, 784])

🖼 **Picture.** Top: random codes decoded by `02`'s autoencoder ($k = 2$), drawn from the box its codes occupy. Bottom: codes drawn from $\mathcal{N}(0, I)$, decoded by the VAE. Every VAE sample is a digit-like image.

In [ ]:
fig_vae_samples()

**A new test image, encoded:** the encoder doesn't return a code, it returns a centre and a spread.

In [ ]:
with torch.no_grad():
    h = vae.encoder(X_te[:1])                   # the first test image (a 7)
    mu, logvar = vae.to_mu(h), vae.to_logvar(h)
print(mu.numpy().round(2), logvar.numpy().round(2), torch.exp(0.5 * logvar).numpy().round(3))
# → [[ 1.9  -2.52]] [[-3.79 -3.33]] [[0.15  0.189]]

So for this 7, the VAE says: "its code is around $(1.90, -2.52)$, give or take about $0.15$ to $0.19$ in each direction". **The rest of this notebook answers:** why a cloud instead of a point (§1–2), how to train something with a random step in the middle (§3), what pulls the codes towards $\mathcal{N}(0, I)$ (§4), and why that loss is the right one (§5).

## 1. The problem: an autoencoder can't generate

> ↩ **Recap** · [02 §4 · the code space ↗](02_autoencoders.ipynb)
>
> With $k = 2$, `02`'s autoencoder put each digit in its own odd-shaped group, with gaps between and around them. A code in a group decoded to a digit; between groups, a smudge; in empty space, a blob.

**What we'd need for generation.** (1) To know **where** valid codes are, so we can sample them: ideally one simple region, the same for every training run. (2) Every code in that region should decode to something sensible, **with no gaps**. The plain autoencoder gives neither: its loss only cares about the codes of training images, one point each.

## 2. The idea: the encoder outputs a distribution

**The idea, in two parts.** (1) Encode each image as a small **cloud** of codes, a normal distribution $\mathcal{N}(\mu, \sigma^2)$, and train the decoder on a **random code from that cloud**. Now the decoder must turn a whole neighbourhood of codes into the same image, so the space around each training code becomes meaningful; neighbouring clouds overlap, and the gaps fill up. (2) Add a penalty that pulls every cloud towards $\mathcal{N}(0, I)$. Then all codes live in one known round region around 0, which is where we'll sample. Picture a classroom: instead of each student getting one exact seat (the AE), each gets a fuzzy area, and the teacher insists everyone sits near the front centre. Areas overlap, so there are no empty seats in between.

🖼 **Picture.** One test image through the trained VAE, with its real numbers. Blue: the encoder and its two outputs; yellow: the random $\varepsilon$; green: the code; orange: the decoder.

In [ ]:
fig_vae_architecture()

**The formulas, piece by piece**, for one image $x$ and a code of size $k$:

$$
h = \operatorname{ReLU}(W_1 x + b_1), \qquad \mu = W_\mu h + b_\mu, \qquad \log \sigma^2 = W_v h + b_v, \qquad z = \mu + \sigma \odot \varepsilon, \qquad \hat x = g(z)
$$

| Symbol | Shape | In words |
|---|---|---|
| $h$ | (256,) | the encoder's hidden layer |
| $\mu$ | $(k,)$ = (2,) | the centre of this image's cloud of codes |
| $\log \sigma^2$ | (2,) | the log of its variance in each direction. Why the log: a linear layer outputs any real number, and $\sigma^2 = e^{\log \sigma^2}$ is then always positive |
| $\varepsilon$ | (2,) | fresh random numbers from $\mathcal{N}(0, 1)$, new for every image at every step |
| $z$ | (2,) | the code used this time: a random point of the cloud. $\odot$ is element-wise multiplication |
| $\hat x = g(z)$ | (784,) | the decoder's output: a probability for each pixel |

**The loss** has two terms, explained in §4 and §5: a **reconstruction** term (the decoded $\hat x$ should match $x$; binary cross-entropy summed over the 784 pixels) and a **KL** term (the cloud $\mathcal{N}(\mu, \sigma^2)$ should be close to $\mathcal{N}(0, I)$).

$$
L(x) = \underbrace{-\sum_{j=1}^{784} \Big[x_j \log \hat x_j + (1 - x_j) \log (1 - \hat x_j)\Big]}_{\text{reconstruction: BCE over the pixels}} \;+\; \underbrace{D_{\text{KL}}\big(\mathcal{N}(\mu, \sigma^2) \,\Vert\, \mathcal{N}(0, I)\big)}_{\text{pull the cloud towards } \mathcal{N}(0, I)}
$$

**The two terms pull against each other.** Reconstruction alone would shrink every cloud to a point (σ → 0, the plain AE) and spread the centres anywhere. KL alone would make every cloud exactly $\mathcal{N}(0, I)$, so the code says nothing about the image. The balance between them is what gives an organized, gap-free code space.

## 3. The reparameterization trick

**The problem.** Training needs $\partial L / \partial \mu$ and $\partial L / \partial \sigma$, to improve the encoder. But between them and the loss sits "draw $z$ from $\mathcal{N}(\mu, \sigma^2)$": a random draw, not a formula, so there's nothing to differentiate. The gradient stops there.

**The idea.** Draw the randomness first, from a distribution that doesn't depend on anything we train, and then **compute** $z$ from it with an ordinary formula:

$$
\varepsilon \sim \mathcal{N}(0, 1), \qquad z = \mu + \sigma\, \varepsilon \quad\Longrightarrow\quad z \sim \mathcal{N}(\mu, \sigma^2), \qquad \frac{\partial z}{\partial \mu} = 1, \qquad \frac{\partial z}{\partial \sigma} = \varepsilon
$$

$z$ has exactly the distribution we wanted (§0's third check), but now it's a plain function of $\mu$ and $\sigma$, with $\varepsilon$ as just another input, like the image. Backprop treats $\varepsilon$ as a constant.

🖼 **Picture.** Left: sampling $z$ directly blocks the gradient. Right: with $z = \mu + \sigma\varepsilon$, the gradient reaches $\mu$ and $\sigma$ through two ordinary derivatives.

In [ ]:
fig_reparam()

**A tiny example by hand.** $\mu = 1.0$, $\sigma = 0.5$, and this time $\varepsilon = -0.4$, so $z = 1.0 + 0.5 \cdot (-0.4) = 0.8$. Take the loss $L = z^2$, so $\partial L / \partial z = 2z = 1.6$. Chain rule: $\partial L / \partial \mu = 1.6 \cdot 1 = 1.6$ and $\partial L / \partial \sigma = 1.6 \cdot \varepsilon = 1.6 \cdot (-0.4) = -0.64$. Autograd agrees:

In [ ]:
mu = torch.tensor(1.0, requires_grad=True)
sigma = torch.tensor(0.5, requires_grad=True)
eps = torch.tensor(-0.4)                        # the random part: an input, no gradient needed
z = mu + sigma * eps
(z ** 2).backward()
print(round(z.item(), 3), round(mu.grad.item(), 3), round(sigma.grad.item(), 3))   # → 0.8 1.6 -0.64

**In the VAE** the encoder outputs $\log \sigma^2$, so the line is `z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)`: $e^{\frac12 \log \sigma^2} = \sigma$, and `randn_like` draws $\varepsilon$ with the same shape as `mu`.

**In one sentence:** write the random code as $z = \mu + \sigma\varepsilon$ with $\varepsilon$ drawn separately, so the random step is an input and gradients flow to $\mu$ and $\sigma$.

## 4. The KL term in closed form

**The question.** The loss needs $D_{\text{KL}}(\mathcal{N}(\mu, \sigma^2) \,\Vert\, \mathcal{N}(0, 1))$ for every image at every step. Week 2's definition is a sum (for continuous distributions, an integral); can it be computed exactly?

**Proposition.** For one dimension,

$$
D_{\text{KL}}\big(\mathcal{N}(\mu, \sigma^2) \,\Vert\, \mathcal{N}(0, 1)\big) = \tfrac{1}{2}\big(\sigma^2 + \mu^2 - 1 - \log \sigma^2\big)
$$

and for a $k$-dimensional code with independent coordinates (a diagonal covariance), the KL is the **sum** of this over the $k$ coordinates. With the encoder's output $\ell = \log \sigma^2$, that's $-\frac12 \sum_j (1 + \ell_j - \mu_j^2 - e^{\ell_j})$, the line you'll see in every VAE's code.

**What it says, in words.** Three pieces: $\mu^2$ punishes a cloud whose centre is far from 0; $\sigma^2 - 1 - \log \sigma^2$ punishes a cloud that's too narrow or too wide (it's 0 at $\sigma^2 = 1$ and positive otherwise); the $\frac12$ is a scale. **Small example:** $\mu = 2$, $\sigma^2 = 1$: $\frac12(1 + 4 - 1 - 0) = 2$. A narrow cloud, $\mu = 0$, $\sigma^2 = 0.09$: $\frac12(0.09 - 1 + 2.408) = 0.749$.

🖼 **Picture.** Four clouds $q = \mathcal{N}(\mu, \sigma^2)$ (purple) against the prior $\mathcal{N}(0, 1)$ (grey), with their KL.

In [ ]:
fig_kl_gaussians()

**The proof in plain words.** KL is the average, under $q$, of $\log q(z) - \log p(z)$. Both are Gaussians, so both logs are quadratics in $z$; the average of a quadratic under a Gaussian only needs its mean and variance, which we know. Everything else cancels.

**The proof.** Let $q = \mathcal{N}(\mu, \sigma^2)$ and $p = \mathcal{N}(0, 1)$. From the log-densities (§0),

$$
\log q(z) - \log p(z) = -\tfrac12 \log \sigma^2 - \frac{(z - \mu)^2}{2\sigma^2} + \frac{z^2}{2} \qquad (\text{the } -\tfrac12 \log 2\pi \text{ terms cancel})
$$

Take the average over $z \sim q$, using $\mathbb{E}_q[(z - \mu)^2] = \sigma^2$ and $\mathbb{E}_q[z^2] = \operatorname{Var}(z) + (\mathbb{E}z)^2 = \sigma^2 + \mu^2$:

$$
D_{\text{KL}}(q \,\Vert\, p) = \mathbb{E}_q\big[\log q(z) - \log p(z)\big] = -\tfrac12 \log \sigma^2 - \frac{\sigma^2}{2\sigma^2} + \frac{\sigma^2 + \mu^2}{2} = \tfrac12\big(\sigma^2 + \mu^2 - 1 - \log \sigma^2\big). \qquad \blacksquare
$$

For independent coordinates, $\log q(z) - \log p(z)$ is a sum over coordinates, so its average is the sum of the one-dimensional KLs. **Checked numerically** for $\mu = 2$, $\sigma^2 = 1$, by averaging over a million samples:

In [ ]:
torch.manual_seed(0)
mu_, var_ = 2.0, 1.0
z = mu_ + var_ ** 0.5 * torch.randn(1_000_000)
q = torch.distributions.Normal(mu_, var_ ** 0.5); p = torch.distributions.Normal(0.0, 1.0)
print(round((q.log_prob(z) - p.log_prob(z)).mean().item(), 3), 0.5 * (var_ + mu_ ** 2 - 1 - np.log(var_)))   # → 1.997 2.0   the average over samples, and the formula

**Remarks.** (1) Minimized at $\mu = 0$, $\sigma^2 = 1$: setting the derivatives to zero, $\partial/\partial \mu = \mu = 0$ and $\partial/\partial \sigma^2 = \frac12(1 - 1/\sigma^2) = 0$. (2) The $-\log \sigma^2$ goes to $+\infty$ as $\sigma \to 0$: a cloud may not collapse to a point, which is exactly what prevents the plain AE's behaviour. (3) This closed form is why the prior and the encoder are both Gaussian: any other choice needs the sampling estimate in the code above, which is noisy.

**In one sentence:** the KL between a Gaussian cloud and the standard normal is $\frac12(\sigma^2 + \mu^2 - 1 - \log \sigma^2)$ per coordinate, smallest when the cloud is the standard normal itself.

## 5. Why this loss: the ELBO

**The question.** §2's loss looks like two sensible terms put together. Is there a reason for exactly this combination? Yes: it's the negative of a lower bound on the log-likelihood, so minimizing it is (approximately) maximum likelihood, the principle behind every loss so far (Week 5 · 02).

**The model.** A generative story for an image: draw a code $z \sim p(z) = \mathcal{N}(0, I)$, then draw each pixel from the decoder's probabilities, $p(x \mid z)$. The likelihood of an image is then $p(x) = \int p(x \mid z)\, p(z)\, dz$: the average over all codes. Maximum likelihood wants $\log p(x)$ large for every training image, but the integral over all codes can't be computed. The encoder's cloud $q(z \mid x) = \mathcal{N}(\mu(x), \sigma^2(x))$ is a guess at which codes could have produced $x$.

**Proposition (the ELBO).** For any distribution $q(z \mid x)$,

$$
\log p(x) \;=\; \underbrace{\mathbb{E}_{q(z \mid x)}\big[\log p(x \mid z)\big] - D_{\text{KL}}\big(q(z \mid x) \,\Vert\, p(z)\big)}_{\text{ELBO}(x)} \;+\; \underbrace{D_{\text{KL}}\big(q(z \mid x) \,\Vert\, p(z \mid x)\big)}_{\ge\, 0} \;\;\ge\;\; \text{ELBO}(x)
$$

**What it says, in words.** The log-likelihood is the ELBO plus a gap, and the gap is how far the encoder's guess $q(z \mid x)$ is from the true posterior $p(z \mid x)$ (which codes really could have made $x$). The gap is never negative, so the ELBO is a **lower bound**. Raising the ELBO either raises $\log p(x)$ or makes the encoder's guess better, both good. And the ELBO is two familiar terms: the first is minus the reconstruction loss (for Bernoulli pixels, $\log p(x \mid z)$ is minus the BCE), the second is §4's KL. **So the VAE's loss is $-\text{ELBO}$.**

**The proof in plain words.** Write $\log p(x)$ as an average over $q$ (it doesn't depend on $z$, so averaging changes nothing), use Bayes' rule $p(x) = p(x, z)/p(z \mid x)$ inside, and multiply and divide by $q$. The log splits into two pieces: one is the ELBO, the other is a KL.

**The proof.** For every $z$, Bayes' rule gives $p(x) = \dfrac{p(x, z)}{p(z \mid x)}$. Therefore

$$
\begin{aligned} \log p(x) &= \mathbb{E}_{q}\Big[\log \frac{p(x, z)}{p(z \mid x)}\Big] = \mathbb{E}_{q}\Big[\log \frac{p(x, z)}{q(z \mid x)} \cdot \frac{q(z \mid x)}{p(z \mid x)}\Big] \\ &= \mathbb{E}_{q}\Big[\log \frac{p(x, z)}{q(z \mid x)}\Big] + \mathbb{E}_{q}\Big[\log \frac{q(z \mid x)}{p(z \mid x)}\Big] = \mathbb{E}_{q}\Big[\log \frac{p(x, z)}{q(z \mid x)}\Big] + D_{\text{KL}}\big(q(z \mid x) \,\Vert\, p(z \mid x)\big). \end{aligned}
$$

Split the first term with $p(x, z) = p(x \mid z)\, p(z)$:

$$
\mathbb{E}_{q}\Big[\log \frac{p(x \mid z)\, p(z)}{q(z \mid x)}\Big] = \mathbb{E}_{q}\big[\log p(x \mid z)\big] - \mathbb{E}_{q}\Big[\log \frac{q(z \mid x)}{p(z)}\Big] = \mathbb{E}_{q}\big[\log p(x \mid z)\big] - D_{\text{KL}}\big(q(z \mid x) \,\Vert\, p(z)\big).
$$

KL divergences are never negative (Week 2), so $\log p(x) \ge \text{ELBO}(x)$. $\blacksquare$

**Remarks.** (1) **Why the expectation becomes one sample:** $\mathbb{E}_q[\log p(x \mid z)]$ is estimated by one $z = \mu + \sigma\varepsilon$ per image per step (§3); over many steps the noise averages out. (2) **The connection to EM (Week 4):** EM's E-step sets $q$ to the exact posterior, making the gap zero; that's possible for a Gaussian mixture, not for a neural decoder. The VAE instead **learns** an encoder that guesses $q$ for any $x$ in one forward pass (called *amortized* inference). (3) **β-VAE:** weighting the KL by $\beta > 1$ gives smoother, more organized codes and blurrier images; $\beta < 1$ the reverse. (4) **Why VAE samples are blurry:** the decoder outputs each pixel's probability, an average over the images that share a code; diffusion models (2020 on) and GANs give sharper samples, but the VAE's ideas (latent codes, the ELBO) are inside many of them.

**In one sentence:** the VAE maximizes the ELBO, reconstruction minus KL, which is a lower bound on the log-likelihood $\log p(x)$, short of it only by how wrong the encoder's guess of the codes is.

## 6. Training, and the latent space

The loss in code, averaged over the batch (both terms are sums over pixels or code dimensions, divided by the batch size):

In [ ]:
def vae_loss_parts(x, x_hat, mu, logvar):
    recon = F.binary_cross_entropy(x_hat, x, reduction="sum") / len(x)          # BCE summed over the pixels
    kl = -0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp()) / len(x)         # §4, summed over the code
    return recon, kl

torch.manual_seed(0)                            # the forward pass draws a random z
with torch.no_grad():
    x_hat, mu, logvar = vae(X_te[:1000])
    recon, kl = vae_loss_parts(X_te[:1000], x_hat, mu, logvar)
print(round(recon.item(), 1), round(kl.item(), 2))     # → 150.2 5.27   per image, on 1000 test images

The training loop that made `vae2.pt` (about 10 seconds): the model returns three things, and the loss needs all of them.

In [ ]:
torch.manual_seed(0)
model = make_vae(2)                              # figures.py: the class of exercise 5 with d=784, h=256, k=2
loader = DataLoader(TensorDataset(X_tr), batch_size=128, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(10):
    model.train()
    for (xb,) in loader:
        x_hat, mu, logvar = model(xb)            # forward: encode, sample z, decode
        recon, kl = vae_loss_parts(xb, x_hat, mu, logvar)
        loss = recon + kl                        # = -ELBO per image
        opt.zero_grad(); loss.backward(); opt.step()
model.eval()
print(all(torch.equal(a, b) for a, b in zip(model.state_dict().values(), vae.state_dict().values())))   # → True   the same weights as vae2.pt

**What the KL term did to the codes.** The same 5,000 test images, as codes of `02`'s autoencoder and as centres $\mu$ of the VAE:

🖼 **Picture.** Left: the autoencoder, codes spread over a large irregular area. Right: the VAE, all centres in one round region around 0, groups touching without gaps.

In [ ]:
fig_vae_latent()

**Every code means something.** Decode a whole grid of codes from $-2.5$ to $2.5$:

🖼 **Picture.** The decoder's output for each code of a 15 × 15 grid. Digits morph smoothly into each other: moving the code a little changes the image a little.

In [ ]:
fig_vae_manifold()

**Remarks.** (1) With $k = 2$ the images are blurry; a VAE with $k = 16$ or $32$ rebuilds far better (but can't be drawn). (2) At test time, to **encode** an image use $\mu$ (no sampling); to **generate**, sample $z \sim \mathcal{N}(0, I)$ and use only the decoder. (3) The reconstruction BCE is about 150 per image and the KL about 5: the KL is small but it's what organized the space.

## 7. Exercises

**Exercise 2 · The reparameterization trick.** Return $z = \mu + \sigma \odot \varepsilon$ where $\sigma = \exp(\frac12 \log\sigma^2)$. All three are tensors of the same shape; `eps` is given (so the check can be exact).

**Example**: what goes in, and what should come out:
```python
reparameterize(torch.tensor([1.0]), torch.tensor([float(np.log(0.25))]), torch.tensor([-0.4]))    # → tensor([0.8000])   1 + 0.5·(−0.4)
```

In [ ]:
def reparameterize(mu, logvar, eps):
    pass  # your code

In [ ]:
check(2)

**Exercise 3 · The KL term, per image.** `mu` and `logvar` are `(N, k)`. Return a tensor `(N,)`: for each row, $-\frac12 \sum_j (1 + \ell_j - \mu_j^2 - e^{\ell_j})$.

**Example**: what goes in, and what should come out:
```python
kl_per_sample(torch.tensor([[1.0, 0.0], [2.0, 0.0]]), torch.zeros(2, 2))    # → tensor([0.5000, 2.0000])   ½·1², ½·2²
```

In [ ]:
def kl_per_sample(mu, logvar):
    pass  # your code

In [ ]:
check(3)

**Exercise 4 · The VAE loss.** `x`, `x_hat` `(N, d)` with `x_hat` in $(0, 1)$; `mu`, `logvar` `(N, k)`. Return one number (a 0-dimensional tensor): (BCE summed over all pixels of the batch + KL summed over the batch) / N. Use `F.binary_cross_entropy(x_hat, x, reduction="sum")`.

**Example**: what goes in, and what should come out:
```python
x = torch.tensor([[1.0, 0.0]]); x_hat = torch.tensor([[0.9, 0.2]])
vae_loss(x, x_hat, torch.zeros(1, 2), torch.zeros(1, 2))    # → tensor(0.3285)   −log 0.9 − log 0.8, KL 0
```

In [ ]:
def vae_loss(x, x_hat, mu, logvar):
    pass  # your code

In [ ]:
check(4)

**Exercise 5 · The VAE as a class.** `VAE(d, h, k)` with these attributes, created **in this order**: `self.encoder = nn.Sequential(nn.Linear(d, h), nn.ReLU())`, `self.to_mu = nn.Linear(h, k)`, `self.to_logvar = nn.Linear(h, k)`, `self.decoder = nn.Sequential(nn.Linear(k, h), nn.ReLU(), nn.Linear(h, d), nn.Sigmoid())`. `forward(x)` returns `(x_hat, mu, logvar)`, with $z$ drawn by `mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)`.

**Example**: what goes in, and what should come out:
```python
x_hat, mu, logvar = VAE(784, 256, 2)(torch.rand(3, 784))
x_hat.shape, mu.shape, logvar.shape    # → (torch.Size([3, 784]), torch.Size([3, 2]), torch.Size([3, 2]))
```

In [ ]:
class VAE(nn.Module):
    def __init__(self, d, h, k):
        pass  # your code

    def forward(self, x):
        pass  # your code

In [ ]:
check(5)

**Recipe (VAE).** Encoder → $\mu$ and $\log\sigma^2$ (two linear heads); $z = \mu + e^{\frac12 \log\sigma^2} \odot \varepsilon$; decoder with a sigmoid; loss = BCE summed over pixels + $-\frac12\sum(1 + \log\sigma^2 - \mu^2 - \sigma^2)$, averaged over the batch. To generate: $z \sim \mathcal{N}(0, I)$, decode. To encode: use $\mu$.

In [ ]:
for n in range(1, 6):
    check(n)

---
Solutions: `solutions/03_solutions.py`. Next: `04_anomaly_detection.ipynb`.